This post is pretty much borrowed with slight modifications from this excellent [blog post](https://gist.github.com/mblondel/586753) from Matthieu Blondel


### General QP problem
A general QP problem can be formulated as follows:

$$
\begin{align*}
\text{minimize} \quad & \frac{1}{2} x^T P x + q^T x \\
\text{subject to} \quad & G x \leq h \\
& A x = b
\end{align*}
$$

### Solving in Dual

In dual in hard margin SVM, the dual problem is formulated as follows:

$$
\begin{align*}
\text{maximize} \quad & \sum_{i=1}^{m} \alpha_i - \frac{1}{2} \sum_{i=1}^{m} \sum_{j=1}^{m} \alpha_i \alpha_j y^{(i)} y^{(j)} (x^{(i)})^T x^{(j)} \\
\text{subject to} \quad & \alpha_i \geq 0 \quad \text{for} \quad i = 1, 2, \ldots, m \\
& \sum_{i=1}^{m} \alpha_i y^{(i)} = 0
\end{align*}
$$

Or, we can write it in the following form (minimization problem):
$$
\begin{align*}
\text{minimize} \quad & \frac{1}{2} \alpha^T Q \alpha + e^T \alpha \\
\text{subject to} \quad & y^T \alpha = 0 \\
& \alpha \geq 0
\end{align*}
$$

where
$$
\begin{align*}
Q_{ij} & = y^{(i)} y^{(j)} (x^{(i)})^T x^{(j)} \\
e_i & = -1
\end{align*}
$$

we can again write it in the following form using kernel trick:

$$
Q_{ij} = y^{(i)} y^{(j)} K(x^{(i)}, x^{(j)})
$$

where $K(x^{(i)}, x^{(j)}) = (x^{(i)})^T x^{(j)}$ is the kernel function.

Thus, we can now set
$$
\begin{align*}
P & = Q \\
q & = e \\
G & = -I \\
h & = 0 \\
A & = y^T \\
b & = 0
\end{align*}
$$

where $I$ is the identity matrix.



In [ ]:
try: 
    import cvxopt
    import cvxopt.solvers

except:
    %pip install cvxopt
    import cvxopt
    import cvxopt.solvers

import numpy as np
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
def linear_kernel(x1, x2):
    return np.dot(x1, x2)

def polynomial_kernel(x, y, p=3):
    return (1 + np.dot(x, y)) ** p

def gaussian_kernel(x, y, sigma=5.0):
    return np.exp(-np.linalg.norm(x-y)**2 / (2 * (sigma ** 2)))

In [ ]:
X, y = make_blobs(n_samples=40, centers=2, n_features=2, random_state=42)

y[y == 0] = -1.0
plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired)

In [ ]:
n_samples, n_features = X.shape
kernel = linear_kernel

# Gram matrix
K = np.zeros((n_samples, n_samples))
for i in range(n_samples):
    for j in range(n_samples):
        K[i,j] = kernel(X[i], X[j])

In [ ]:
# QP problem formulation
P = cvxopt.matrix(np.outer(y, y) * K)
q = cvxopt.matrix(-np.ones(n_samples))
A = cvxopt.matrix(y, (1, n_samples), 'd')
b = cvxopt.matrix(0.0)
G = cvxopt.matrix(-np.eye(n_samples))
h = cvxopt.matrix(np.zeros(n_samples))

In [ ]:
dictionary = {"P": P, "q": q, "A": A, "b": b, "G": G, "h": h}
for key, val in dictionary.items():
    print(key, val.size)

In [ ]:
solution = cvxopt.solvers.qp(P, q, G, h, A, b)

In [ ]:
alphas_sol = np.array(solution['x']).flatten()
alphas_sol

In [ ]:
alphas_sol[alphas_sol > 1e-5]

In [ ]:
# Support vectors have non zero lagrange multipliers
sv = alphas_sol > 1e-5
ind = np.arange(len(alphas_sol))[sv]
a = alphas_sol[sv]
sv_x = X[sv]
sv_y = y[sv]

# Plot the support vectors
plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired)

# Mark the support vectors with thick black circles
plt.scatter(sv_x[:, 0], sv_x[:, 1], c=sv_y, s=100, cmap=plt.cm.Paired, marker='o', edgecolors='k')

In [ ]:
w = np.sum(a * sv_y[:, None] * sv_x, axis=0)
w

In [ ]:
b = np.sum(sv_y - np.dot(sv_x, w)) / len(sv_y)
b

In [ ]:
# Plot the decision boundary and margins
plt.scatter(X[:, 0], X[:, 1], c=y, s=30, cmap=plt.cm.Paired)
plt.scatter(sv_x[:, 0], sv_x[:, 1], c=sv_y, s=100, cmap=plt.cm.Paired, marker='o', edgecolors='k')

x_min = np.min(X[:, 0])
x_max = np.max(X[:, 0])

y_min = (-w[0] * x_min - b) / w[1]
y_max = (-w[0] * x_max - b) / w[1]

plt.plot([x_min, x_max], [y_min, y_max], 'k-')

# Margin 1
y_min = (-w[0] * x_min - b + 1) / w[1]
y_max = (-w[0] * x_max - b + 1) / w[1]

plt.plot([x_min, x_max], [y_min, y_max], 'k--')

# Margin 2
y_min = (-w[0] * x_min - b - 1) / w[1]
y_max = (-w[0] * x_max - b - 1) / w[1]

plt.plot([x_min, x_max], [y_min, y_max], 'k--')



### Using cvxpy

In [ ]:
import cvxpy as cp

alpha = cp.Variable(n_samples)

inequality_constraints = [alpha >= 0]
equality_constraints = [cp.multiply(alpha, y) == 0]

overall_constraints = inequality_constraints + equality_constraints
P_numpy = np.outer(y, y)* K   + 3e-8 * np.eye(n_samples)
objective = cp.Maximize(cp.sum(alpha) - 0.5 * cp.quad_form(alpha, P_numpy))

prob = cp.Problem(objective, overall_constraints)
result = prob.solve()

In [ ]:
prob.status
alpha.value.sort()
alpha.value

### Primal using cvxopt

In [ ]:
# Number of features
n_features = X.shape[1]

# Identity matrix
P = cvxopt.matrix(np.block([[np.eye(n_features), np.zeros((n_features, 1))], [np.zeros((1, n_features + 1))]]))

# Inequality constraints matrix
G = cvxopt.matrix(-np.block([np.diag(y) @ X, y.reshape(-1, 1)]))

# Inequality constraints bounds
h = cvxopt.matrix(-np.ones(X.shape[0]))

# Solve the QP problem
solution = cvxopt.solvers.qp(P, cvxopt.matrix(np.zeros(n_features + 1)), G, h)

# Extract optimized w and b values
w_opt = np.array(solution['x'][:-1])
b_opt = np.array(solution['x'][-1])

In [ ]:
w_opt, w

In [ ]:
b_opt, b